# 05 - Explore cycling-facility records

**Source:** `amenagements-velo-en-ile-de-france.csv` (full file by default).
**Module:** `scripts/datasets/cycling.py`.

Inspect highway categories, street/commune labels, OSM IDs, lengths and geometry text.
This source contains more than conventional cycleway rows. Classification codes such
as `ad`, `ag` and `nv` require their source documentation before interpretation.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.io as pio
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Open the hackathon project folder before running this notebook.')
if not Path(sys.executable).resolve().is_relative_to(ROOT / '.venv'):
    raise RuntimeError('Select the project .venv kernel, then restart and run all.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.datasets import cycling
from scripts.exploration import column_summary, repeated_keys, save_parquet

pio.renderers.default = 'plotly_mimetype'
print('Python:', sys.executable)


Python: d:\Hackathon Mobilités 2026 Spécial Dataviz\.venv\Scripts\python.exe


In [2]:
source = ROOT / cycling.FILENAME
raw = cycling.load(source)  # Add rows=10_000 to explore a first-row sample instead.


In [3]:
print('Loaded rows and columns:', raw.shape)
display(raw.head())
raw.info()


Loaded rows and columns: (151130, 17)


,geo_shape,osm_id,nom_com,nom_voie,insee_com,highway,sens_voit,revetement,panneaux,ad,ag,nv,longueur,notes,petite_ech,moyenn_ech,geo_point_2d
0,"{""coordinates"": [[2.4468091, 48.86839609998634...",696377163.0,Montreuil,Rue des Hanots,93048,living_street,UNIQUE,asphalt,,,DSC,z20,149,,3,33,"48.86897181246889, 2.4470897899560935"
1,"{""coordinates"": [[2.4605116, 48.87060039998634...",38127073.0,Montreuil,Rue Jules Verne,93048,unclassified,UNIQUE,asphalt,,,DSC,,153,,3,34,"48.870971493565136, 2.459629812440381"
2,"{""coordinates"": [[2.457569, 48.87017269998635]...",23305199.0,Montreuil,Rue Saint-Victor,93048,living_street,DOUBLE,asphalt,,,,z20,271,,3,33,"48.87095775087993, 2.4561609302828664"
3,"{""coordinates"": [[2.4601348, 48.86832139998636...",1107037943.0,Montreuil,Rue de Rosny,93048,footway,NC,,,piste trottoir uni,piste trottoir uni,hors voirie,153,,1,11,"48.86861415649596, 2.461078419570594"
4,"{""coordinates"": [[2.4625035, 48.86886749998635...",25791137.0,Montreuil,Rue de la Nouvelle France,93048,residential,UNIQUE,,,,DSC,limite 30,355,,3,32,"48.86729532009446, 2.4621047241656986"


<class 'pandas.DataFrame'>
RangeIndex: 151130 entries, 0 to 151129
Data columns (total 17 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   geo_shape     151130 non-null  string
 1   osm_id        151130 non-null  string
 2   nom_com       151130 non-null  string
 3   nom_voie      151130 non-null  string
 4   insee_com     151130 non-null  string
 5   highway       151130 non-null  string
 6   sens_voit     151130 non-null  string
 7   revetement    151130 non-null  string
 8   panneaux      151130 non-null  string
 9   ad            151130 non-null  string
 10  ag            151130 non-null  string
 11  nv            151130 non-null  string
 12  longueur      151130 non-null  string
 13  notes         151130 non-null  string
 14  petite_ech    151130 non-null  string
 15  moyenn_ech    151130 non-null  string
 16  geo_point_2d  151130 non-null  string
dtypes: string(17)
memory usage: 72.5 MB


## Clean a copy and inspect the columns

The original file and `raw` table stay unchanged. Strip surrounding spaces and turn
blank cells into missing values; preserve special codes such as `NA` and `ZZ`.
Dataset-specific numeric conversions are documented in the module's `clean()` function.
Unexpected numeric text raises an error for inspection. No rows are dropped or deduplicated.


In [4]:
cleaned = cycling.clean(raw)
assert len(cleaned) == len(raw)
display(cleaned.head())
display(column_summary(cleaned))


,geo_shape,osm_id,nom_com,nom_voie,insee_com,highway,sens_voit,revetement,panneaux,ad,ag,nv,longueur,notes,petite_ech,moyenn_ech,geo_point_2d,latitude,longitude,coordinates_in_bounds
0,"{""coordinates"": [[2.4468091, 48.86839609998634...",696377163.0,Montreuil,Rue des Hanots,93048,living_street,UNIQUE,asphalt,<NA>,<NA>,DSC,z20,149,<NA>,3,33,"48.86897181246889, 2.4470897899560935",48.868972,2.44709,True
1,"{""coordinates"": [[2.4605116, 48.87060039998634...",38127073.0,Montreuil,Rue Jules Verne,93048,unclassified,UNIQUE,asphalt,<NA>,<NA>,DSC,<NA>,153,<NA>,3,34,"48.870971493565136, 2.459629812440381",48.870971,2.45963,True
2,"{""coordinates"": [[2.457569, 48.87017269998635]...",23305199.0,Montreuil,Rue Saint-Victor,93048,living_street,DOUBLE,asphalt,<NA>,<NA>,<NA>,z20,271,<NA>,3,33,"48.87095775087993, 2.4561609302828664",48.870958,2.456161,True
3,"{""coordinates"": [[2.4601348, 48.86832139998636...",1107037943.0,Montreuil,Rue de Rosny,93048,footway,NC,<NA>,<NA>,piste trottoir uni,piste trottoir uni,hors voirie,153,<NA>,1,11,"48.86861415649596, 2.461078419570594",48.868614,2.461078,True
4,"{""coordinates"": [[2.4625035, 48.86886749998635...",25791137.0,Montreuil,Rue de la Nouvelle France,93048,residential,UNIQUE,<NA>,<NA>,<NA>,DSC,limite 30,355,<NA>,3,32,"48.86729532009446, 2.4621047241656986",48.867295,2.462105,True


,data_type,missing_rows,distinct_values
column,,,
geo_shape,string,59,151071
osm_id,string,0,142451
nom_com,string,0,1052
nom_voie,string,42171,27087
insee_com,string,0,1055
highway,string,0,17
sens_voit,string,0,3
revetement,string,38033,33
panneaux,string,147959,38


## Analyse categories and measurements

Length becomes numeric; its unit still needs metadata confirmation. OSM IDs, including
any trailing `.0`, stay as text. No segments are merged. `geo_shape` stays JSON text.
`geo_point_2d` is parsed as latitude/longitude; missing points remain missing.


In [5]:
facility_summary = cycling.facility_summary(cleaned)
display(facility_summary)
display(cleaned['longueur'].describe())
display(cleaned.loc[~cleaned.coordinates_in_bounds].head(10))
display(repeated_keys(cleaned, ['osm_id']).head(20))


,highway,facility_records
7,residential,53436
0,cycleway,17345
1,footway,13864
12,tertiary,11598
3,path,11007
8,secondary,10368
10,service,6807
5,primary,6235
2,living_street,6097
4,pedestrian,5943


count      151130.0
mean     112.207411
std       157.09771
min             0.0
25%            19.0
50%            60.0
75%           145.0
max          3984.0
Name: longueur, dtype: Float64

,geo_shape,osm_id,nom_com,nom_voie,insee_com,highway,sens_voit,revetement,panneaux,ad,ag,nv,longueur,notes,petite_ech,moyenn_ech,geo_point_2d,latitude,longitude,coordinates_in_bounds
28106,<NA>,4293956.0,8e Arrondissement,Avenue Pierre-I??-de-Serbie,75108,residential,DOUBLE,asphalt,<NA>,<NA>,<NA>,limite 30,0,<NA>,3,32,<NA>,<NA>,<NA>,False
28110,<NA>,181731946.0,6e Arrondissement,Quai Voltaire,75106,service,UNIQUE,asphalt,<NA>,voie bus uni,<NA>,limite 30,0,<NA>,2,24,<NA>,<NA>,<NA>,False
28117,<NA>,69236724.0,10e Arrondissement,Rue Gabriel Laumain,75110,residential,UNIQUE,asphalt,<NA>,<NA>,DSC,z30,0,<NA>,3,31,<NA>,<NA>,<NA>,False
28118,<NA>,521065516.0,9e Arrondissement,Rue Bleue,75109,cycleway,UNIQUE,asphalt,<NA>,piste uni,<NA>,hors voirie,0,<NA>,1,12,<NA>,<NA>,<NA>,False
28119,<NA>,4018042.0,1er Arrondissement,Avenue de l'Opéra,75101,secondary,DOUBLE,asphalt,<NA>,voie bus uni,voie bus uni,limite 30,0,<NA>,2,23,<NA>,<NA>,<NA>,False
28121,<NA>,10662531.0,9e Arrondissement,Impasse d'Amsterdam,75109,service,UNIQUE,paving_stones,<NA>,chemin service site propre uni,<NA>,hors voirie,0,<NA>,1,12,<NA>,<NA>,<NA>,False
28122,<NA>,32785178.0,9e Arrondissement,Rue de Londres,75109,secondary,UNIQUE,sett,<NA>,<NA>,voie bus uni,limite 30,0,<NA>,2,24,<NA>,<NA>,<NA>,False
28123,<NA>,860117924.0,10e Arrondissement,Rue de Dunkerque,75110,residential,UNIQUE,asphalt,<NA>,<NA>,DSC bande,limite 30,0,<NA>,2,22,<NA>,<NA>,<NA>,False
28207,<NA>,8599647.0,16e Arrondissement,Rue Marcel Dassault,75116,residential,UNIQUE,asphalt,<NA>,<NA>,<NA>,limite 30,0,<NA>,3,32,<NA>,<NA>,<NA>,False
28311,<NA>,336799982.0,20e Arrondissement,Avenue Simón Bolívar,75120,secondary,UNIQUE,asphalt,<NA>,<NA>,<NA>,z30,0,<NA>,3,31,<NA>,<NA>,<NA>,False


,osm_id,records
8086,109878869.0,20
98837,46195218.0,16
82322,34491643.0,13
4277,105940431.0,13
56999,226850767.0,13
128580,800601465.0,13
135007,89165029.0,13
82067,34287257.0,12
130395,827027182.0,11
68261,270125070.0,11


In [ ]:
cycling.plot_facilities(facility_summary).show()
cycling.plot_locations(cleaned, max_points=5_000).show()

## Play with classification codes

Choose a highway category and inspect its other codes. The coordinate plot samples
representative points, not line geometry or a routable network. Neither a valid point
nor a category alone establishes safe, legal cycling access. Tables show category counts
and coordinates independently of the charts.


In [7]:
HIGHWAY = 'cycleway'
selected = cleaned.loc[cleaned.highway.eq(HIGHWAY)]
display(selected[['osm_id', 'nom_com', 'nom_voie', 'highway', 'ad', 'ag', 'nv', 'longueur']].head(20))
display(cleaned[['nom_com', 'nom_voie', 'latitude', 'longitude']].head(20))


,osm_id,nom_com,nom_voie,highway,ad,ag,nv,longueur
20,938209774.0,Juvisy-sur-Orge,<NA>,cycleway,piste uni,<NA>,hors voirie,51
25,1381089987.0,Draveil,<NA>,cycleway,piste uni,<NA>,hors voirie,15
31,219672118.0,Saint-Chéron,<NA>,cycleway,piste uni,piste uni,hors voirie,799
32,545220421.0,Saint-Chéron,<NA>,cycleway,piste uni,<NA>,hors voirie,32
58,586416235.0,Wissous,<NA>,cycleway,piste uni,piste uni,hors voirie,634
59,1373006570.0,Wissous,<NA>,cycleway,piste uni,piste uni,hors voirie,5
61,89451784.0,Montreuil,<NA>,cycleway,piste uni,piste uni,hors voirie,120
65,1209373835.0,Montreuil,<NA>,cycleway,piste uni,<NA>,velorue,40
79,480535393.0,Viry-Châtillon,<NA>,cycleway,piste uni,piste uni,hors voirie,53
82,148882456.0,Bruyères-le-Châtel,<NA>,cycleway,piste uni,piste uni,hors voirie,473


,nom_com,nom_voie,latitude,longitude
0,Montreuil,Rue des Hanots,48.868972,2.44709
1,Montreuil,Rue Jules Verne,48.870971,2.45963
2,Montreuil,Rue Saint-Victor,48.870958,2.456161
3,Montreuil,Rue de Rosny,48.868614,2.461078
4,Montreuil,Rue de la Nouvelle France,48.867295,2.462105
5,Montreuil,Rue des Jardins Dufour,48.86735,2.459981
6,Montreuil,Rue de la Nouvelle France,48.864757,2.461997
7,Montreuil,<NA>,48.863778,2.462744
8,Montreuil,<NA>,48.857306,2.464124
9,Montreuil,<NA>,48.856634,2.463096


## Optional: save this dataset as Parquet



In [8]:
SAVE_CLEANED = False
output_path = ROOT / 'data' / 'cleaned' / 'cycling.parquet'

if SAVE_CLEANED:
    save_parquet(cleaned, output_path)
    print('Saved', len(cleaned), 'rows to', output_path)
else:
    print('No cleaned data written. Change SAVE_CLEANED after reviewing this dataset.')


No cleaned data written. Change SAVE_CLEANED after reviewing this dataset.
